In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

d = pd.read_csv('data PM10 mean 2025.csv', sep=";")

# Filtrer la France métropolitaine
d_metro = d[
    (d["validité"] == 1) &
    (d["Latitude"].between(41.0, 51.5)) &
    (d["Longitude"].between(-5.5, 10.0))
].dropna(subset=["Latitude", "Longitude", "valeur"])

In [3]:
print(d_metro["type d'implantation"].value_counts())
print()
print(d_metro["type d'influence"].value_counts())

type d'implantation
Urbaine                   211
Périurbaine                51
Rurale régionale           15
Rurale près des villes     12
Rurale nationale            9
Name: count, dtype: int64

type d'influence
Fond            203
Trafic           62
Industrielle     33
Name: count, dtype: int64


In [6]:
import libpysal
from spreg import ML_Lag

# Encodage des variables qualitatives
d_sar = d_metro.copy()

implantation_dummies = pd.get_dummies(d_sar["type d'implantation"], drop_first=True).astype(int)
influence_dummies = pd.get_dummies(d_sar["type d'influence"], drop_first=True).astype(int)

d_sar = pd.concat([d_sar, implantation_dummies, influence_dummies], axis=1)

# Variable dépendante et covariables
noms = implantation_dummies.columns.tolist() + influence_dummies.columns.tolist()
y = d_sar["valeur"].values.reshape(-1, 1)
X = d_sar[noms].values

# Matrice de voisinage
coords_sar = list(zip(d_sar["Longitude"], d_sar["Latitude"]))
W = libpysal.weights.KNN.from_array(np.array(coords_sar), k=8)
W.transform = "r"

# Modèle SAR avec noms de variables
sar = ML_Lag(y, X, W, name_y="PM10", name_x=noms)
print(sar.summary)

ML_Lag
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :        PM10                Number of Observations:         298
Mean dependent var  :     15.1040                Number of Variables   :           8
S.D. dependent var  :      3.3694                Degrees of Freedom    :         290
Pseudo R-squared    :      0.5070
Spatial Pseudo R-squared:  0.3788
Log likelihood      :   -684.3786
Sigma-square ML     :      5.5891                Akaike info criterion :    1384.757
S.E of regression   :      2.3641                Schwarz criterion     :    1414.334

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
-------------------------------------